# 🚀 SOTA E-Commerce Visual Search Engine (Colab Edition)

Notebook ini akan mengunduh repositori arsitektur mesin pencari canggih (FastAPI + Qdrant + CLIP + React) yang telah kita bangun, mengonfigurasi environment di Google Colab, menjalankan server backend & frontend di latar belakang, dan memberikan Anda URL publik melalui **Localtunnel** agar Anda bisa langsung menguji purwarupa UI aplikasi ini.

---

## Langkah 1: Persiapan Environment & Download Source Code

In [ ]:
# Ganti YOUR_USERNAME/YOUR_REPOSITORY sesuai dengan repo tempat Anda mengunggah kodenya.
# Contoh: !git clone https://github.com/my-username/Smart-Catalog-Amazon-Berkeley-Objects.git /content/sota_search_engine
!git clone https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git /content/sota_search_engine

# PENTING: Instalasi pip dengan mengabaikan dependensi yang konflik secara aman.
!pip install fastapi uvicorn torch transformers qdrant-client>=1.10.0 Pillow numpy opencv-python optuna segment-anything python-multipart urllib3>=2 --ignore-installed

# Install Localtunnel untuk mengekspos localhost ke public web
!npm install -g localtunnel

## Langkah 2: Restart Runtime (PENTING)
**Perhatian:** Karena adanya pembaharuan library sistem (seperti Protobuf dan Urllib), Anda **wajib** melakukan restart runtime Colab sekarang.

Pilih menu: `Runtime` -> `Restart session`, lalu jalankan cell di bawah ini.

In [ ]:
import os
import subprocess
import time

try:
    import qdrant_client
    print(f"Qdrant Client Version: {qdrant_client.__version__}")
except Exception as e:
    print("Tolong pastikan Anda sudah melakukan Restart Runtime Colab.")
    raise e

# Update sedikit script di core.py untuk menyesuaikan dengan qdrant-client versi modern jika dibutuhkan
import re
core_path = "/content/sota_search_engine/backend/core.py"
if os.path.exists(core_path):
    with open(core_path, 'r') as f:
        content = f.read()
    
    # Memastikan penggunaan get_collection/query_points aman di versi modern
    content = re.sub(r'self.client.search\(', 'self.client.query_points(', content)
    content = re.sub(r'return search_result', 'return getattr(search_result, "points", search_result)', content)
    
    with open(core_path, 'w') as f:
        f.write(content)

# Jalankan Backend di background
backend_process = subprocess.Popen(
    ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/sota_search_engine/backend"
)
print("Memulai Backend FastAPI di port 8000...")
time.sleep(15)  # Tunggu model selesai loading

## Langkah 3: Mengindeks Dataset ABO untuk Testing
Mari kita muat beberapa sampel dataset Amazon Berkeley Objects (ABO) ke dalam Qdrant agar bisa langsung dicari.

In [ ]:
import requests
from PIL import Image
import io
import urllib.request

# URL contoh gambar sepatu dari dataset ABO atau dummy images
test_images = {
    "abo_shoe_1": "https://m.media-amazon.com/images/I/7184pWb46WL._AC_UY575_.jpg",  # Black Leather Formal
    "abo_shoe_2": "https://m.media-amazon.com/images/I/61NlH2e12xL._AC_UY575_.jpg",  # White Canvas Sneaker
    "abo_shoe_3": "https://m.media-amazon.com/images/I/71nI6P8ZtBL._AC_UY575_.jpg"   # Red Running Shoe
}

print("Mulai mengindeks gambar ke Vector DB...")
for img_id, url in test_images.items():
    try:
        # Download gambar
        response = requests.get(url)
        img_bytes = io.BytesIO(response.content)
        
        # Post ke API Indexing Backend
        files = {'file': (f'{img_id}.jpg', img_bytes, 'image/jpeg')}
        data = {'id': img_id}
        res = requests.post("http://localhost:8000/index_image", files=files, data=data)
        print(f"[{res.status_code}] Berhasil mengindeks {img_id}")
    except Exception as e:
        print(f"Gagal memproses {img_id}: {e}")

## Langkah 4: Menjalankan Frontend React & Ekspos ke Internet

Karena arsitektur React tidak dirancang untuk ditenagai langsung melalui Cell IPython, kita akan menjalankan proses Node JS di latar belakang dan mempublikasikan URL-nya. 

**Penting:** Klik link Localtunnel yang muncul pada output di bawah untuk mengakses aplikasi!

In [ ]:
# Install npm deps untuk React
print("Install dependensi Frontend...")
!cd /content/sota_search_engine/frontend && npm install --legacy-peer-deps --silent

# Build Frontend agar ter-compile menjadi static HTML
print("Membangun Frontend (Membutuhkan sekitar 1 menit)...")
!cd /content/sota_search_engine/frontend && npm run build > /dev/null 2>&1

# Install server statis sederhana Python
frontend_process = subprocess.Popen(
    ["python3", "-m", "http.server", "3000"],
    cwd="/content/sota_search_engine/frontend/build"
)
print("Frontend berjalan di port 3000...")

# Menggunakan Localtunnel untuk mendapatkan Public URL
print("\n=======================================================")
print("MENDAPATKAN PUBLIC URL...")
print("Tunggu beberapa detik, lalu klik tautan Localtunnel di bawah.")
print("\nPassword Tunnel (jika diminta):")
!wget -q -O - https://loca.lt/mytunnelpassword
print("\n=======================================================")
!lt --port 3000